# Model Evaluation / 模型评估
Go through the process of evaluating a model. / 走一遍完整的模型评估流程。

# Load Data / 加载数据

In [12]:
# Import all necessary modules / 导入所有必要模块
import pandas as pd
import torch
import os
from torch.utils.data import Dataset
from PIL import Image
from torchvision.transforms import v2
from torch.utils.data import DataLoader

In [50]:
# Define transformations / 定义图像变换
test_transform = v2.Compose([
    v2.Resize((128, 128)),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], 
                 std=[0.229, 0.224, 0.225])
])

In [57]:
# Define dataset class and label encoding / 定义数据集类与标签编码
label_encoding = {"malignant": 0, "benign": 1}

# Dataset class / 数据集类
class CustomImageDataset(Dataset):
    def __init__(self, annotations_file, img_dir, transform, target_transform):
        self.img_labels = pd.read_csv(annotations_file)
        self.img_dir = img_dir
        self.transform = transform
        self.target_transform = lambda y: target_transform[y]

    def __len__(self):
        return len(self.img_labels)

    def __getitem__(self, idx):
        from torchvision.io import read_image

        # Skip unreadable/missing images by trying next sample / 跳过无法读取或缺失的图像
        max_tries = len(self.img_labels)
        for _ in range(max_tries):
            img_path = os.path.join(self.img_dir, self.img_labels.iloc[idx, 0])
            label_name = self.img_labels.iloc[idx, 1]
            try:
                image = read_image(img_path)  # Tensor [C,H,W], uint8
                if image.shape[0] == 1:
                    image = image.repeat(3, 1, 1)
                image = self.transform(image)
                label = self.target_transform(label_name)
                return image, label
            except Exception:
                idx = (idx + 1) % len(self.img_labels)

        raise RuntimeError('No readable images found in dataset / 数据集中没有可读取的图像')

In [58]:
# Create dataset / 创建数据集
# Use absolute paths to avoid cwd issues / 使用绝对路径避免工作目录问题
test_dataset = CustomImageDataset(
    annotations_file=r'E:\02_Projects\22_Kode_Kloud\PyTorch\section_3\labs\030-150-evaluate-our-models\test_data.csv',
    img_dir=r'E:\02_Projects\22_Kode_Kloud\PyTorch',
    transform=test_transform,
    target_transform=label_encoding
)

In [59]:
# Create dataloader / 创建数据加载器
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Load Model / 加载模型

In [60]:
# Import required modules / 导入所需模块
import torch.nn as nn
import torch.nn.functional as F

# Define class / 定义模型类
class BreastCancerClassification(nn.Module):
   def __init__(self):
       super(BreastCancerClassification, self).__init__()
       # Create layers / 创建网络层
       self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
       self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
       self.pool = nn.MaxPool2d(2, 2)
      
       self.fc1 = nn.Linear(64 * 32 * 32, 256)
       self.fc2 = nn.Linear(256, 128)
       self.fc3 = nn.Linear(128, 2)

   def forward(self, x):
       # Define layer flow / 定义层间连接流程
       x = self.pool(F.relu(self.conv1(x)))
       x = self.pool(F.relu(self.conv2(x)))
       x = x.view(-1, 64 * 32 * 32)
       x = F.relu(self.fc1(x))
       x = F.relu(self.fc2(x))
       x = self.fc3(x)
       return x


In [61]:
# Create model instance / 创建模型实例
model = BreastCancerClassification()

In [62]:
# Optional: generate breast-cancer checkpoint from lab if missing / 可选：若缺失则从 lab 生成乳腺肿瘤检查点
from pathlib import Path
import runpy
import os
import torch
import torch.nn as nn
import torch.optim as optim

lab_dir = (Path.cwd().parent.parent / 'labs' / '030-060-train-and-save-an-image-classi').resolve()
project_root = (Path.cwd().parent.parent.parent).resolve()  # .../PyTorch
print(f"Lab dir / Lab目录: {lab_dir}")
print(f"Project root / 项目根目录: {project_root}")

if not lab_dir.exists():
    raise FileNotFoundError(f"Lab directory not found / 未找到Lab目录: {lab_dir}")

existing = sorted(lab_dir.glob('*_checkpoint.tar'))
if existing:
    print('✓ Existing checkpoint(s) in lab / Lab中已有检查点:')
    for p in existing:
        print(f' - {p.name}')
else:
    print('No lab checkpoint found, trying training_loop.py ... / 未找到检查点，尝试运行训练脚本...')
    old_cwd = Path.cwd()
    lab_success = False
    try:
        os.chdir(project_root)
        runpy.run_path(str(lab_dir / 'training_loop.py'), run_name='__main__')
        lab_success = True
    except Exception as e:
        print(f"⚠ Lab training failed / Lab训练失败: {e}")
    finally:
        os.chdir(old_cwd)

    generated = sorted(lab_dir.glob('*_checkpoint.tar'))
    if generated:
        lab_success = True

    if lab_success:
        print('✓ Generated checkpoint(s) in lab / Lab中已生成检查点:')
        for p in generated:
            print(f' - {p.name}')
    else:
        print('Fallback: quick bootstrap training on current test_loader / 回退：使用当前 test_loader 快速训练生成检查点')
        model.train()
        criterion_bootstrap = nn.CrossEntropyLoss()
        optimizer_bootstrap = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

        running_loss = 0.0
        max_batches = 10
        for batch_idx, (inputs, labels) in enumerate(test_loader):
            optimizer_bootstrap.zero_grad()
            outputs = model(inputs)
            loss = criterion_bootstrap(outputs, labels)
            loss.backward()
            optimizer_bootstrap.step()
            running_loss += loss.item()
            if batch_idx + 1 >= max_batches:
                break

        bootstrap_path = Path.cwd() / 'bootstrap_breast_cancer_checkpoint.tar'
        torch.save({
            'epoch': 0,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer_bootstrap.state_dict(),
            'train_loss': running_loss / max_batches
        }, bootstrap_path)

        print(f"✓ Bootstrap checkpoint generated / 已生成回退检查点: {bootstrap_path}")

Lab dir / Lab目录: E:\02_Projects\22_Kode_Kloud\PyTorch\section_3\labs\030-060-train-and-save-an-image-classi
Project root / 项目根目录: E:\02_Projects\22_Kode_Kloud\PyTorch
No lab checkpoint found, trying training_loop.py ... / 未找到检查点，尝试运行训练脚本...
⚠ Lab training failed / Lab训练失败: 'JpegImageFile' object has no attribute '_im'
Fallback: quick bootstrap training on current test_loader / 回退：使用当前 test_loader 快速训练生成检查点
✓ Bootstrap checkpoint generated / 已生成回退检查点: e:\02_Projects\22_Kode_Kloud\PyTorch\section_3\demos\030-135-model-evaluation\bootstrap_breast_cancer_checkpoint.tar


In [55]:
import torch
from pathlib import Path

# Search candidate checkpoint files / 搜索候选检查点文件
base_dir = Path.cwd()
candidate_dirs = [
    base_dir,
    base_dir.parent / '030-045-saving-and-loading-models',
    base_dir.parent / '030-060-train-and-save-an-image-classi',
    base_dir.parent / '030-105-additional-training-methods',
]

candidate_files = []
for d in candidate_dirs:
    if d.exists():
        for pattern in ('*.tar', '*.pt', '*.pth'):
            candidate_files.extend(sorted(d.glob(pattern)))

print('Candidate files / 候选文件:')
for p in candidate_files:
    print(f' - {p}')

# Match checkpoint with current model architecture / 匹配当前模型结构
model_keys = set(model.state_dict().keys())
matched_checkpoint_path = None
matched_state_dict = None

for path in candidate_files:
    try:
        obj = torch.load(path, map_location='cpu', weights_only=False)

        if isinstance(obj, dict) and 'model_state_dict' in obj:
            state_dict = obj['model_state_dict']
        elif isinstance(obj, dict) and all(isinstance(v, torch.Tensor) for v in obj.values()):
            state_dict = obj
        elif hasattr(obj, 'state_dict'):
            state_dict = obj.state_dict()
        else:
            continue

        ckpt_keys = set(state_dict.keys())
        if ckpt_keys == model_keys:
            matched_checkpoint_path = path
            matched_state_dict = state_dict
            print(f"\n✓ Matched checkpoint / 匹配成功: {path}")
            break
    except Exception as e:
        print(f"Skip {path.name} -> {e}")

if matched_checkpoint_path is None:
    raise FileNotFoundError(
        'No compatible checkpoint found for BreastCancerClassification. '
        '请先在 section_3/labs/030-060-train-and-save-an-image-classi 训练并生成 *_checkpoint.tar，'
        '或把兼容的检查点放到当前目录。'
    )

Candidate files / 候选文件:
 - e:\02_Projects\22_Kode_Kloud\PyTorch\section_3\demos\030-135-model-evaluation\bootstrap_breast_cancer_checkpoint.tar
 - e:\02_Projects\22_Kode_Kloud\PyTorch\section_3\demos\030-045-saving-and-loading-models\5_checkpoint.tar
 - e:\02_Projects\22_Kode_Kloud\PyTorch\section_3\demos\030-045-saving-and-loading-models\training_checkpoint_0.tar
 - e:\02_Projects\22_Kode_Kloud\PyTorch\section_3\demos\030-045-saving-and-loading-models\training_checkpoint_2.tar
 - e:\02_Projects\22_Kode_Kloud\PyTorch\section_3\demos\030-045-saving-and-loading-models\training_checkpoint_4.tar
 - e:\02_Projects\22_Kode_Kloud\PyTorch\section_3\demos\030-045-saving-and-loading-models\training_checkpoint_6.tar
 - e:\02_Projects\22_Kode_Kloud\PyTorch\section_3\demos\030-045-saving-and-loading-models\training_checkpoint_8.tar
 - e:\02_Projects\22_Kode_Kloud\PyTorch\section_3\demos\030-045-saving-and-loading-models\training_checkpoint_final.tar
 - e:\02_Projects\22_Kode_Kloud\PyTorch\section_3

In [63]:
# Load parameters into model / 将参数加载到模型
model.load_state_dict(matched_state_dict)
print(f"✓ Loaded checkpoint / 已加载检查点: {matched_checkpoint_path}")

✓ Loaded checkpoint / 已加载检查点: e:\02_Projects\22_Kode_Kloud\PyTorch\section_3\demos\030-135-model-evaluation\bootstrap_breast_cancer_checkpoint.tar


# Model Inference / 模型推理

## Real-time / 实时推理
Single-input inference for immediate response. / 单输入推理，响应更及时。

In [66]:
# Open an image / 打开一张图片
# Reimport PIL.Image to avoid ultralytics patches / 重新导入 PIL.Image 以避免 ultralytics 补丁
import PIL.Image
from torchvision.io import read_image

image_path = 'sample-input.jpg'
image = read_image(image_path)  # Read as tensor
if image.shape[0] == 1:
	image = image.repeat(3, 1, 1)  # Convert grayscale to RGB

In [67]:
# Set eval mode / 设置评估模式
model.eval()

BreastCancerClassification(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=65536, out_features=256, bias=True)
  (fc2): Linear(in_features=256, out_features=128, bias=True)
  (fc3): Linear(in_features=128, out_features=2, bias=True)
)

In [68]:
%%time
# Apply transform / 应用变换
transformed_image = test_transform(image)

# Perform inference / 执行推理
output = model(transformed_image)
_, predicted = torch.max(output.data, 1) # Get max score class / 取分数最高的类别

# Print class / 打印类别
print(f"Class / 类别: {predicted.item()}") # item() extracts scalar / item() 提取标量

Class / 类别: 0
CPU times: total: 0 ns
Wall time: 10.5 ms


In [69]:
# Reverse-map label_encoding / 反向映射标签编码
index_to_class_map = {v: k for k, v in label_encoding.items()}
print(f"Class / 类别: {index_to_class_map[predicted.item()]}")

Class / 类别: malignant


## Batch Inference / 批量推理
Execute multiple inputs at once. / 一次执行多个输入。

This is the form commonly used in training/evaluation loops. / 这是训练/评估循环中最常用的方式。

PyTorch DataLoader naturally works in batches. / PyTorch DataLoader 天然支持批处理。

In [70]:
# Show dataloader and dataset / 显示数据加载器与数据集信息
print(f"batch size / 批大小: {test_loader.batch_size}")
print(f"dataset images / 数据集图片数: {len(test_dataset)}")

batch size / 批大小: 32
dataset images / 数据集图片数: 100


In [71]:
# Use dataloader for batch inference / 使用数据加载器执行批量推理
for i, data in enumerate(test_loader, 0): # Loop in batches of 32 / 按32的批次遍历
    inputs, labels = data
    
    # Perform inference as above / 与前面相同方式执行推理
    outputs = model(inputs)
    _, predicted = torch.max(outputs.data, 1)
    print(f"Class / 类别: {predicted}") # no .item() for batch tensor / 批量张量无需 .item()

Class / 类别: tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])
Class / 类别: tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])
Class / 类别: tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])
Class / 类别: tensor([0, 0, 0, 0])


# torch.no_grad() / torch.no_grad()
Run inference without gradient computation. / 在不计算梯度的情况下执行推理。

Lower resource usage, often faster. / 资源开销更小，通常更快。

In [72]:
%%time
# Try again with timing / 再运行一次并计时
for i, data in enumerate(test_loader, 0):
    inputs, labels = data
    outputs = model(inputs)
    _, predicted = torch.max(outputs.data, 1)
    print(f"Class / 类别: {predicted}")

Class / 类别: tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])
Class / 类别: tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])
Class / 类别: tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])
Class / 类别: tensor([0, 0, 0, 0])
CPU times: total: 2.05 s
Wall time: 388 ms


In [73]:
%%time
# Same with no_grad() / 使用 no_grad() 的同等流程
with torch.no_grad():
    for i, data in enumerate(test_loader, 0):
        inputs, labels = data
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        
        print(f"Class / 类别: {predicted}")

Class / 类别: tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])
Class / 类别: tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])
Class / 类别: tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])
Class / 类别: tensor([0, 0, 0, 0])
CPU times: total: 1.88 s
Wall time: 382 ms


# Torchmetrics / Torchmetrics
Torchmetrics integrates well with PyTorch and provides ready-to-use metrics. / Torchmetrics 与 PyTorch 集成良好，提供开箱即用的评估指标。

In [74]:
# Import module / 导入模块
import torchmetrics

In [86]:
# Simulate torchmetrics setup / 模拟 torchmetrics 设置
num_classes = 2  # Binary classification: malignant vs benign / 二分类：恶性 vs 良性

In [87]:
# Create sample inputs / 创建示例输入
inputs = torch.tensor([
    [2.0, 1.0],   # Class 0 (malignant) / 类别0（恶性）
    [0.5, 2.5],   # Class 1 (benign) / 类别1（良性）
    [2.1, 0.5],   # Class 0
    [1.0, 0.8],   # Class 0
    [0.3, 1.5],   # Class 1
    [0.9, 2.1],   # Class 1
    [0.2, 3.0],   # Class 1
    [2.1, 0.1],   # Class 0
    [0.5, 2.5],   # Class 1
    [1.0, 2.0]    # Class 1
 ])

In [88]:
# Create true labels (first and last intentionally incorrect) / 创建真实标签（首尾样本故意设置为错误）
true_labels = torch.tensor([1, 1, 0, 0, 1, 1, 1, 0, 1, 0])  # Binary labels: 0=malignant, 1=benign / 二分类标签：0=恶性，1=良性

In [89]:
# Initialize multiclass accuracy metric / 初始化多分类准确率指标
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=num_classes) # Multiclass task / 多分类任务

In [90]:
# Simulate predictions on inputs / 在输入上模拟预测
_, predicted = torch.max(inputs.data, 1)

In [91]:
# Update metric with predictions and labels / 用预测与标签更新指标
accuracy.update(predicted, true_labels) 

In [92]:
# Compute final accuracy / 计算最终准确率
final_accuracy = accuracy.compute()
print(f"Accuracy / 准确率: {final_accuracy * 100}%")  

Accuracy / 准确率: 80.0%


# Test Loop / 测试循环
Run model evaluation in a test loop and use Torchmetrics for metric computation. / 在测试循环中执行模型评估，并使用 Torchmetrics 计算指标。

For this, use no_grad() and ensure the model is in eval mode. / 同时使用 no_grad() 并确保模型处于 eval 模式。

In [93]:
import torchmetrics

# Initialize accuracy metric / 初始化准确率指标
accuracy_metric = torchmetrics.Accuracy(task="binary", num_classes=2)  # Binary classification / 二分类任务

# Test loop with Torchmetrics / 使用 Torchmetrics 的测试循环
model.eval()  # Set eval mode / 设置评估模式
with torch.no_grad():  # Disable gradients for evaluation / 评估时关闭梯度
    # Iterate test dataloader / 遍历测试数据加载器
    for i, data in enumerate(test_loader, 0):
        inputs, labels = data
        outputs = model(inputs)

        # Get predicted class / 获取预测类别
        _, predicted = torch.max(outputs.data, 1)

        # Update metric / 更新指标
        accuracy_metric.update(predicted, labels)

# Compute final accuracy / 计算最终准确率
final_accuracy = accuracy_metric.compute()
print(f"Accuracy / 准确率: {final_accuracy * 100:.2f}%")  # Format to 2 decimal places / 保留2位小数


Accuracy / 准确率: 50.00%


In [94]:
# NOTE: Reset metric before reuse; otherwise values keep accumulating. / 注意：复用前要重置指标，否则会持续累加。
accuracy_metric.reset()

# Custom Metric Example / 自定义指标示例

In [95]:
# Initialize correct and total counts / 初始化正确数与总数
correct = 0
total = 0

In [96]:
# Run predictions / 执行预测
with torch.no_grad():
    for i, data in enumerate(test_loader, 0):
        inputs, labels = data
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)

        # Update total by label count / 按标签数量更新总数
        total += labels.size(0)
        # Update correct count when prediction matches label / 预测与标签一致时累加正确数
        correct += (predicted == labels).sum().item()

# Accuracy percentage = 100 * correct / total / 准确率百分比 = 100 * correct / total
print(f'Accuracy / 准确率: {100 * correct / total:.2f}%')  # Use float division for precision / 使用浮点除法保持精度

Accuracy / 准确率: 50.00%
